In [1]:
import requests
import time
import pandas as pd
from datetime import datetime

import os
import yaml
from tqdm import tqdm  # New import for the progress bar

from pathlib import Path

In [2]:
root = Path.cwd().parent
root


PosixPath('/workspaces/Binance-Predict-Value-And-Trend')

In [3]:
def datetime_to_ms(date_str):
    """Convert a datetime string to millisecond timestamp."""
    if pd.isna(date_str):
        return None
    if '.' in str(date_str):
        dt_obj = datetime.strptime(str(date_str), "%Y-%m-%d %H:%M:%S.%f")
    else:
        dt_obj = datetime.strptime(str(date_str), "%Y-%m-%d %H:%M:%S")
    return int(dt_obj.timestamp() * 1000)

def ms_to_datetime(ms_timestamp):
    """Convert millisecond timestamp back to readable string."""
    return datetime.fromtimestamp(ms_timestamp / 1000.0).strftime('%Y-%m-%d %H:%M:%S')

def load_config(config_path="config.yaml"):
    """Load the YAML configuration file."""
    with open(config_path, 'r') as file:
        return yaml.safe_load(file)

def fetch_and_save_binance_data(config_path="config.yaml"):
    """
    Fetch Binance data, resume from the last saved timestamp,
    and stream directly into a CSV file with a clean progress bar.
    """
    config = load_config(config_path)
    symbol = config['data']['symbol']
    timeframe = config['data']['time']['frame']
    start_date_cfg = config['data']['time']['start']
    end_date_cfg = config['data']['time']['end']
    save_dir = root / "data"/ "raw"

    base_url = "https://data-api.binance.vision/api/v3/klines"
    limit = 1000

    end_ts = datetime_to_ms(end_date_cfg)
    columns = [
        'Open_time', 'Open', 'High', 'Low', 'Close', 'Volume',
        'Close_time', 'Quote_asset_volume', 'Number_of_trades',
        'Taker_buy_base_asset_volume', 'Taker_buy_quote_asset_volume', 'Ignore'
    ]

    file_path = save_dir / f"{symbol}_{timeframe}_data_v2.csv"
    start_ts = None

    # 1. INCREMENTAL LOAD CHECK
    if os.path.exists(file_path) and os.path.getsize(file_path) > 0:
        try:
            existing_df = pd.read_csv(file_path, usecols=['Close_time'])
            if not existing_df.empty:
                last_close_time_str = existing_df.iloc[-1]['Close_time']
                start_ts = datetime_to_ms(last_close_time_str) + 1
                # Use tqdm.write instead of print to prevent breaking the layout if needed later
                print(f"🔄 Resuming download from timestamp: {last_close_time_str}")
        except Exception as e:
            print(f"⚠️ Error reading existing file: {e}. Starting over.")

    if start_ts is None:
        start_ts = datetime_to_ms(start_date_cfg)
        print(f"🚀 Creating new stream. Starting from config date: {start_date_cfg}")

    if start_ts >= end_ts:
        print("✅ Data is already fully updated up to the requested end_date.")
        return

    # 2. PROGRESS BAR SETUP
    total_duration = end_ts - start_ts

    # Initialize tqdm
    pbar = tqdm(total=total_duration, desc=f"Downloading {symbol}", unit="ms",
                bar_format="{l_bar}{bar}| {percentage:3.0f}% [{elapsed}<{remaining}] {postfix}")

    current_ts = start_ts
    total_fetched = 0
    write_header = not os.path.exists(file_path)

    while current_ts < end_ts:
        params = {
            "symbol": symbol,
            "interval": timeframe,
            "startTime": current_ts,
            "endTime": end_ts,
            "limit": limit
        }

        max_retries = 5
        retry_count = 0
        success = False
        data = []

        while retry_count < max_retries and not success:
            try:
                response = requests.get(base_url, params=params, timeout=15)
                response.raise_for_status()
                data = response.json()
                success = True

            except Exception as e:
                # Use tqdm.write to print errors without breaking the progress bar
                tqdm.write(f"⚠️ [Error] {e}. Retrying ({retry_count+1}/{max_retries})...")
                retry_count += 1
                time.sleep(5 * retry_count)

        if not success or not data:
            tqdm.write("🚨 Failed to fetch data or no more data available.")
            break

        # Process batch
        batch_df = pd.DataFrame(data, columns=columns)
        batch_df['Open_time'] = pd.to_datetime(batch_df['Open_time'], unit='ms')
        batch_df['Close_time'] = pd.to_datetime(batch_df['Close_time'], unit='ms')

        numeric_cols = ['Open', 'High', 'Low', 'Close', 'Volume', 'Quote_asset_volume',
                        'Taker_buy_base_asset_volume', 'Taker_buy_quote_asset_volume']
        batch_df[numeric_cols] = batch_df[numeric_cols].apply(pd.to_numeric, axis=1)
        batch_df = batch_df.drop(columns=['Ignore'])

        # Append to CSV
        batch_df.to_csv(file_path, mode='a', index=False, header=write_header)
        write_header = False
        total_fetched += len(batch_df)

        # 3. UPDATE PROGRESS BAR & POSTFIX (Latest Time)
        last_kline_close_time = data[-1][6]
        latest_time_str = ms_to_datetime(last_kline_close_time)

        step_duration = (last_kline_close_time + 1) - current_ts

        # SỬA Ở ĐÂY: Cài đặt chữ hiển thị trước, nhưng KHÔNG vẽ lại (refresh=False)
        pbar.set_postfix_str(f"Latest Fetch: {latest_time_str}", refresh=False)

        # Gọi update() để nó tiến hành cộng dồn thời gian và vẽ lại màn hình 1 lần duy nhất
        pbar.update(step_duration)

        current_ts = last_kline_close_time + 1
        time.sleep(0.2)

    pbar.close()

    print("\n=========================================")
    print(f"🎉 DOWNLOAD COMPLETED! Added {total_fetched} new candles.")
    print(f"📂 Data safely streamed to: {file_path}")
    print("=========================================")

if __name__ == "__main__":
    CONFIG_FILE_PATH = "/workspaces/Binance-Predict-Value-And-Trend/config.yaml"
    try:
        fetch_and_save_binance_data(config_path=CONFIG_FILE_PATH)
    except FileNotFoundError:
        print("🚨 Error: config.yaml not found.")

🚀 Creating new stream. Starting from config date: 2024-07-01 00:00:00



🎉 DOWNLOAD COMPLETED! Added 1008000 new candles.
📂 Data safely streamed to: /workspaces/Binance-Predict-Value-And-Trend/data/raw/BTCUSDT_1m_data_v2.csv


In [9]:
df = pd.read_csv("/workspaces/Binance-Predict-Value-And-Trend/data/raw/BTCUSDT_1m_data_v2.csv")
df.head(10)

,Open_time,Open,High,Low,Close,Volume,Close_time,Quote_asset_volume,Number_of_trades,Taker_buy_base_asset_volume,Taker_buy_quote_asset_volume
0,2024-07-01 00:00:00,62772.01,62809.01,62772.00,62776.01,24.35543,2024-07-01 00:00:59.999,1.529470e+06,1215,14.75592,9.266116e+05
1,2024-07-01 00:01:00,62776.00,62786.00,62768.00,62769.72,8.85121,2024-07-01 00:01:59.999,5.556372e+05,837,3.05137,1.915461e+05
2,2024-07-01 00:02:00,62769.72,62781.55,62760.78,62760.79,20.12564,2024-07-01 00:02:59.999,1.263276e+06,1175,11.17319,7.013346e+05
3,2024-07-01 00:03:00,62760.78,62781.60,62745.65,62745.66,19.23422,2024-07-01 00:03:59.999,1.207179e+06,1039,10.66982,6.696609e+05
4,2024-07-01 00:04:00,62745.66,62745.66,62722.07,62722.08,20.52790,2024-07-01 00:04:59.999,1.287725e+06,918,2.03293,1.275226e+05
5,2024-07-01 00:05:00,62722.08,62798.95,62722.07,62780.98,101.04709,2024-07-01 00:05:59.999,6.341254e+06,1575,90.86127,5.701925e+06
6,2024-07-01 00:06:00,62780.97,62780.98,62748.02,62750.58,13.91924,2024-07-01 00:06:59.999,8.736484e+05,1687,5.27516,3.310780e+05
7,2024-07-01 00:07:00,62750.58,62772.00,62750.57,62758.11,20.66490,2024-07-01 00:07:59.999,1.296947e+06,5218,11.60757,7.284696e+05
8,2024-07-01 00:08:00,62758.11,62758.11,62713.00,62713.01,16.48252,2024-07-01 00:08:59.999,1.034083e+06,865,2.31496,1.452465e+05
9,2024-07-01 00:09:00,62713.01,62713.01,62648.48,62648.49,43.46920,2024-07-01 00:09:59.999,2.724852e+06,2201,6.34022,3.973921e+05


In [ ]:
import plotly.graph_objects as go
import pandas as pd

def plot_binance_candles(df):
    """
    Vẽ biểu đồ nến tương tác với phong cách giao diện Binance.
    """
    # Đảm bảo các cột định dạng đúng
    df['Open_time'] = pd.to_datetime(df['Open_time'])

    # Khởi tạo biểu đồ nến
    fig = go.Figure(data=[go.Candlestick(
        x=df['Open_time'],
        open=df['Open'],
        high=df['High'],
        low=df['Low'],
        close=df['Close'],
        increasing_line_color='#26a69a',  # Màu xanh của Binance
        decreasing_line_color='#ef5350'   # Màu đỏ của Binance
    )])

    # Tùy chỉnh giao diện (Layout) giống Binance
    fig.update_layout(
        title='Binance Price Chart',
        yaxis_title='Price (USDT)',
        xaxis_title='Date',
        template='plotly_dark',  # Nền tối
        xaxis_rangeslider_visible=False, # Ẩn thanh trượt bên dưới để tập trung biểu đồ
        paper_bgcolor='#131722',  # Màu nền giấy giống Binance
        plot_bgcolor='#131722',   # Màu nền biểu đồ
        margin=dict(l=10, r=10, t=40, b=10)
    )

    # Thêm khả năng tương tác (phóng to, di chuyển)
    fig.show()

# Giả sử bạn đã có biến 'final_df' từ bước cào dữ liệu:
plot_binance_candles(df)

In [10]:
from autots import AutoTS

# Chuyển đổi cột thời gian sang định dạng datetime
df['Open_time'] = pd.to_datetime(df['Open_time'])

# Đặt cột Open_time làm Index (Yêu cầu bắt buộc hoặc cấu hình trong AutoTS)
df.set_index('Open_time', inplace=True)

# Lựa chọn mục tiêu dự báo: Thông thường ta sẽ dự báo giá đóng cửa ('Close')
# Tuy nhiên, AutoTS cũng hỗ trợ dự báo đa biến (nhiều cột cùng lúc) nếu muốn.
data_target = df[['Close']] 

# 2. KHỞI TẠO MÔ HÌNH AUTOTS
model = AutoTS(
    forecast_length=15,        # Dự báo bao nhiêu bước tiếp theo (ở đây là 15 phút tới)
    frequency='1min',          # Tần suất dữ liệu là 1 phút ('1min' hoặc 'T')
    prediction_interval=0.95,  # Khoảng tin cậy 95%
    ensemble='auto',           # Tự động kết hợp các mô hình tốt nhất (Ensemble)
    model_list='fast',         # Sử dụng danh sách các mô hình chạy nhanh (ARIMA, Theta, ETS, v.v.)
                               # Bạn có thể đổi thành 'default' hoặc 'superfast' tùy cấu hình máy
    transformer_list='fast',   # Các bước biến đổi/làm sạch dữ liệu nhanh
    max_generations=5,         # Số lượng thế hệ thuật toán di truyền để tối ưu tham số
    num_validations=2,         # Số lần chia tập test để kiểm thử chéo (Validation)
    no_negatives=True,         # Đảm bảo giá dự báo không bị âm
    n_jobs='auto'              # Sử dụng tối đa số nhân CPU có sẵn
)

# 3. HUẤN LUYỆN MÔ HÌNH
# AutoTS sẽ tự động thử nghiệm nhiều thuật toán và chọn ra cấu hình tốt nhất
model = model.fit(data_target)

# In ra thông tin mô hình chiến thắng (Best Model)
print("Mô hình tốt nhất được chọn:")
print(model)

# 4. DỰ BÁO TƯƠNG LAI
prediction = model.predict()

# Lấy bảng kết quả dự báo điểm (Point Forecast)
forecast_df = prediction.forecast
print("\nKết quả dự báo giá Close trong 15 phút tiếp theo:")
print(forecast_df)

# (Tùy chọn) Lấy khoảng dự báo trên và dưới
# upper_forecast = prediction.upper_forecast
# lower_forecast = prediction.lower_forecast

# 5. XEM LỊCH SỬ THỬ NGHIỆM CỦA CÁC MÔ HÌNH (Optional)
# Bảng ghi lại điểm số (MAE/RMSE) của tất cả các mô hình đã chạy thử
model_results = model.results()

Using 1 cpus for n_jobs.
Data frequency is: min, used frequency is: 1min
Model Number: 1 with model AverageValueNaive in generation 0 of 5
Model Number: 2 with model AverageValueNaive in generation 0 of 5
Model Number: 3 with model AverageValueNaive in generation 0 of 5
Model Number: 4 with model DatepartRegression in generation 0 of 5
Model Number: 5 with model ETS in generation 0 of 5
Model Number: 6 with model GLM in generation 0 of 5


: 